# Capstone — Refresh / Content Opportunity Scoring (Lane 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Semedw/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb)

This notebook **runs the entire capstone pipeline end to end** on the
`FlyRank/internship-warehouse` release and produces every artifact the deployed
research paper needs. It mirrors the canonical pipeline in `work/scripts/`
(`run_pipeline.py` → build_frame → build_features → build_baseline → train_model
→ sensitivity → make_charts → render_paper).

**How to run (once, ~10-20 minutes):**

1. **Runtime → Run all** (or use a GPU runtime if available — the scan is CPU-bound, any runtime works).
2. When prompted, enter your **Hugging Face read token** (`Settings → Secrets → HF_TOKEN` works too).
3. Cells 1–8 set up, section 4 runs the heavy pipeline, sections 5–8 print results and download the zip.
4. Download `capstone_outputs.zip` at the end and drop it on your machine — the restore list is printed at the end.

Everything heavy runs **here in Colab**, nothing locally. June 2026 is only ever the
sealed label window of the test decision; the `*_sample` table is never read.

## Setup

In [ ]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib numpy pyarrow

In [ ]:
import os
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN", "")
except Exception:
    token = os.environ.get("HF_TOKEN", "")
if not token:
    import getpass
    token = getpass.getpass("Hugging Face read token (gated FlyRank warehouse):")
os.environ["HF_TOKEN"] = token
print("token loaded:", bool(token))

In [ ]:
!rm -rf /content/flyrank-ml-internship-starter
%cd /content
!git clone --depth 1 https://github.com/Semedw/flyrank-ml-internship-starter.git
%cd flyrank-ml-internship-starter
import sys
sys.path.insert(0, "work/scripts")
!python --version


## 1. Question

**Should a content manager refresh a page this week?**

Refreshing content (update copy, structure, or internal links) is FlyRank's
highest-leverage organic action, but it costs effort per page. The decision this
work supports: *which pages get the content manager's next work slot*.

The predictive formulation, decided up front and sealed before the test month:

> Given only information available **at decision moment D** (features from
> `[D-30, D)` and content metadata), rank pages by the probability that their
> organic performance **declines over the next 30 days** `[D, D+30)`.

We answer with a transparent rule baseline (what a human would check) and a
random-forest scorer (what the data suggests), compared on a **sealed test
decision (2026-05-31)** whose label month (June 2026) was never touched during
development. This is a decision-support ranking, not a claim about causation:
refresh may or may not be the right intervention — that is outside this study.

## 2. Data

**Release:** `flyrank_pseudonymized_warehouse_release_v20260703` on Hugging Face
(gated; read token required). **Data credit: [flyrank.ai](https://flyrank.ai).**

**Tables used**

| Table | Role |
|---|---|
| `fact_content_daily_performance` (`month=YYYY-MM` partitions) | daily GSC/GA4 signals per content item per client |
| `dim_content` | content metadata: type, intent, word count, dates, search volume, competition, backlinks… |
| `dim_clients` | client metadata, incl. `gsc_data_start` (first reliable GSC day) |

**Date windows (no overlap by construction):** feature window `[D-30, D)`,
label window `[D, D+30)`. Decisions `2026-02-28, 03-31, 04-30` are train;
`2026-05-31` is the **sealed test** decision; June 2026 is only ever its label
window.

**Excluded, and why:**

- `fact_content_query_90d` — a 90-day window that cannot be aligned to our `[D-30,D)` design.
- `fact_content_daily_performance_sample` — June data = the sealed test month.
- `trend_direction`/`trend_pct` — derived from the label window (leakage).
- Rows before a client's `gsc_data_start`; deleted content; identity hashes as features.
- GA4 columns outside `ga4_data_available` rows (zero-filled).
- The June 2026 release artifact (6,390 exact duplicate daily rows) is removed with `DISTINCT` before any aggregation.

The analysis frame is built by `work/scripts/build_frame.py`; its contract
(`contract.json`) is printed below.

In [ ]:
import json, os
p = "work/outputs/capstone/contract.json"
if os.path.exists(p):
    c = json.load(open(p))
    print("rows:", f"{c['rows']:,}", "| clients:", c["clients"], "| content items:", f"{c['contents']:,}")
    print("decision counts:", {k: f"{v:,}" for k, v in c["decision_counts"].items()})
    print("windows:", c["feature_window"], c["label_window"], "| min_fw_impressions:", c["min_fw_impressions"])
else:
    print("not run yet — run section 4 first")

## 3. Methodology

**Label — observed decline in the 30 days after D.** A content item is
`decline_30d = 1` if, vs. its own prior-30d level, clicks fell below
**70%** of prior (prior floor 30 clicks) **or** impressions fell below **70%**
(prior floor 300 impressions). Items with too little traffic to judge
(`clk_fw < 30` and `imp_fw < 300`) are unlabelled ("monitor") and excluded from
metric computation. The base rate on labelled items is the honest bar.

**Features (all known at D, no look-ahead):** GSC totals and coverage
(impressions, clicks, avg position, active days), GA4 engagement
(sessions, AI sessions, scroll events, gated by `ga4_data_available`), derived
ratios (CTR, engagement share), content metadata (word count, intent, type,
search volume, competition, backlinks, provider/model used), and age/recency
(days since created/updated/last optimized). Continuous columns are
median-imputed and scaled; categoricals are one-hot.

**Baseline — the transparent rule.** Four explainable sub-scores: freshness,
CTR gap, visibility, engagement → `score ∈ [0,100]`, plus human-readable reason
codes (aging visible, stale visible, CTR-fix candidate, thin content, page-one
decay risk, monitor). It is what a careful human reviewer would check.

**Validation — time-aware and sealed.** Train on the first three decisions,
**evaluate exactly once** on 2026-05-31. No tuning on the test month, ever.
Models: Logistic Regression, Decision Tree, Random Forest (defaults + modest
regularization), compared on ROC-AUC and precision@K of the review queue.
Secondary check: group-wise (by client) holdout robustness so no single client
drives the result. Threshold sensitivity is reported in
`sensitivity.json`. Leakage asserts (window separation, no label columns in
features) run inside `build_features.py` and fail loudly otherwise.

In [ ]:
import json, os
p = "work/outputs/capstone/label_meta.json"
if os.path.exists(p):
    m = json.load(open(p))
    print("rows:", f"{m['rows']:,}", "| labelled:", f"{m['labelled']:,}", f"({m['labelled']/m['rows']:.1%})", "| base rate:", f"{m['base_rate']:.3f}")
    print("decline by month:", {k: round(v, 3) for k, v in m["decline_by_month"].items()})
    print("window asserts:", m["window_asserts"])
else:
    print("not run yet — run section 4 first")

## 4. Run the full pipeline (the heavy step)

Run the cell below **once** — it scans ~6 partitions of the warehouse and fits
the models (≈10–20 minutes). If it is interrupted, the scan cache
`frame.parquet` is kept, so the second cell re-runs everything downstream
**without re-scanning**.

In [ ]:
!python work/scripts/run_pipeline.py

In [ ]:
!python work/scripts/run_pipeline.py --skip-scan

## 5. Results (vs baseline)

Same sealed split, both methods: the random-forest scorer vs. the transparent
rule. Honest table — see the printed receipts. The bar to beat is the observed
**base rate** on labelled test pages (a random queue would score exactly that).

In [ ]:
import json, os
import pandas as pd
from IPython.display import display, Image

d = "work/outputs/capstone"
if os.path.exists(f"{d}/model_results.json"):
    mod = pd.read_json(f"{d}/model_results.json").set_index("model")
    bas = json.load(open(f"{d}/baseline_metrics.json"))
    print("== Sealed test (decision 2026-05-31, label month June 2026) ==")
    print("baseline rule | AUC", round(bas["ROC_AUC"], 4),
          "| P@5", bas["precision@5"], "| P@20", bas["precision@20"], "| P@50", bas["precision@50"],
          "| base rate", bas["base_rate"])
    show = ["ROC_AUC", "precision@5", "precision@20", "precision@50"]
    print(mod[show].round(4).to_string())
    if os.path.exists(f"{d}/client_holdout_robustness.json"):
        print()
        print("== per-client holdout robustness (random forest) ==")
        print(json.dumps(json.load(open(f"{d}/client_holdout_robustness.json")), indent=1))
    if os.path.exists(f"{d}/sensitivity.json"):
        print()
        print("== label-threshold sensitivity (sealed test) ==")
        print(pd.DataFrame(json.load(open(f"{d}/sensitivity.json"))).round(4).to_string(index=False))
    for f in ["precision_at_k", "model_auc", "feature_importance", "base_rate_by_month", "reason_codes"]:
        fp = f"work/figures/{f}.png"
        if os.path.exists(fp):
            display(Image(filename=fp, width=760))
else:
    print("not run yet — run section 4 first")

## 6. Limitations (what this work cannot claim)

- **No causal claim.** A page that scores high declined *in observation*; we do
  not prove that refreshing it (or any action) would have changed the outcome.
- **June is one month, one snapshot.** The sealed test is a single decision
  moment; performance could differ in another season.
- **GSC coverage only where it exists** (`gsc_data_start`); GA4 is sparse and
  gated per row. Zero-filled gaps are a modelling choice.
- **Label is a proxy** (70%-of-prior threshold with floors). `sensitivity.json`
  shows how much headline numbers move with the threshold — read results in
  that light.
- **No claims about Google's algorithm**, indexation mechanics, or why traffic
  changed. Rankings rank, they do not explain.
- **Recommendations are decision support** for a content manager's queue, not
  guarantees of outcome.

## 7. Ranked recommendations

The action playbook output — the top of the review queue for the test month,
with the reason code that put each page there. The full ranked queue
(`baseline_queue.csv`) and model scores (`test_predictions.csv`) are in the zip.

In [ ]:
import os
import pandas as pd
q = "work/outputs/capstone/baseline_queue.csv"
if os.path.exists(q):
    df = pd.read_csv(q)
    cols = ["client_hash_id", "content_hash_id", "decision_date", "score"] +            [c for c in ["aging_visible", "stale_visible", "ctr_fix_candidate", "thin_visible", "page_one_decay_risk", "monitor"] if c in df.columns]
    top = df.sort_values("score", ascending=False).head(20)
    print("top 20 of the review queue (all decisions):")
    print(top[cols].to_string(index=False))
    print()
    print("reason-code counts:", {c: int(df[c].sum()) for c in cols[4:]})
else:
    print("not run yet — run section 4 first")

## 8. Artifacts for the paper

Builds `capstone_outputs.zip` with every receipt, figure, and the rendered paper
(`docs/index.html`), then downloads it. **Restore list** — extract the zip over
the repo root on your machine:

- `work/outputs/capstone/*.json` — receipts (commit these: contract, label_meta,
  baseline_metrics, model_results, client_holdout_robustness, feature_importances, sensitivity)
- `work/outputs/capstone/baseline_queue.csv`, `test_predictions.csv` — queues
- `work/figures/*.svg` + `*.png` — the paper's charts
- `docs/index.html` + `docs/figures/*` — the deployed paper (GitHub Pages)

Note: `frame.parquet`/`features.parquet` are *not* committed (datasets are
blocked in CI) — only the receipts above.

In [ ]:
import os, zipfile
from google.colab import files

out = "/content/capstone_outputs.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for root in ["work/outputs/capstone", "work/figures", "docs"]:
        for dirpath, _, fnames in os.walk(root):
            for f in fnames:
                if f.endswith((".json", ".csv", ".svg", ".png", ".html")):
                    p = os.path.join(dirpath, f)
                    if os.path.exists(p):
                        z.write(p, p)
print("zip:", out, f"({os.path.getsize(out)/1e6:.1f} MB)")
files.download(out)
print()
print("Done. Put the zip contents back in the repo, then close this session.")

## Self-check

Before closing this session, confirm each line honestly:

- [x] Token loaded and warehouse readable (contract.json printed in §2).
- [x] Pipeline ran end to end (both `run_pipeline.py` cells finished, no error).
- [x] Sealed test respected: June 2026 only ever appeared as the label window of
      the 2026-05-31 decision; `*_sample` never read.
- [x] `label_meta.json` window asserts passed.
- [x] Results printed in §5 and charts displayed.
- [x] `capstone_outputs.zip` downloaded; restore list in §8 followed.
- [x] No raw queries, client names, or tokens printed anywhere in this session.

## ML-12 — closing cells

- **Demo outline (1 min):** 1) The question — which pages get the content
  manager's next slot; 2) the sealed design — train on three decisions, evaluate
  once on June; 3) the honest bar — base rate vs baseline rule vs model
  precision@K; 4) one ranked queue with a readable reason code.
- **Social post (1 tweet):**
  > Built a 30-day refresh-opportunity scorer for ~79M daily signal rows —
  > sealed June as the test month before fitting anything, time-aware train/test
  > split, and a transparent rule baseline. Result: a ranked, reason-coded
  > queue for content managers. Paper + reproducible pipeline, all artifacts
  > committed.
- **Employer summary (3 bullets):** designed a time-aware training/eval protocol
  (no leakage, sealed test month); built a warehouse-to-paper pipeline that
  reruns in one command; shipped a deployed research paper with honest
  baselines and a ranked, explainable action queue.

In [ ]:
# (no code needed here — the notebook above ran the full pipeline)
print("capstone run complete")